PR Reviewer Tester


In [4]:
from dotenv import load_dotenv
from langgraph.graph import END, START, StateGraph
from state import ReviewState
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [5]:
MAX_ATTEMPTS = 2

# Nodes (all stubs for now)

def fetch_diff(state: ReviewState) -> dict:
    """TODO: fetch the real diff from the GitHub API using repo + pr_number."""
    return {
        "files": [
            {
                "path": "utils.py",
                "diff": (
                    "-def apply_discount(amount: float, percent: float) -> float:\n"
                    "+def apply_discount(amount: float, rate: float) -> float:\n"
                    "-    if percent < 0 or percent > 100:\n"
                    "+    if rate < 0 or rate > 1:\n"
                ),
            }
        ]
    }


def retrieve(state: ReviewState) -> dict:
    """TODO: search the vector store and a name index for related code."""
    context = {
        f["path"]: [
            "service.py: discounted = apply_discount(subtotal, discount_percent)"
        ]
        for f in state["files"]
    }
    return {"context": context}


def reviewer(state: ReviewState) -> dict:
    """TODO: call the LLM with diff + context + tools, return structured comments."""
    comments = [
        {
            "path": "utils.py",
            "line": 4,
            "severity": "bug",
            "issue": "apply_discount now expects a 0 to 1 rate, but service.py "
                     "still passes a percentage like 10, so orders will raise ValueError.",
            "suggestion": "Update calculate_total to pass discount_percent / 100.",
        }
    ]
    return {"draft_comments": comments, "attempts": state.get("attempts", 0) + 1}


def verifier(state: ReviewState) -> dict:
    """TODO: second LLM checks each comment against the code and drops weak ones."""
    kept = [c for c in state["draft_comments"] if c["severity"] != "nit"]
    return {"final_comments": kept, "needs_retry": len(kept) == 0}


def post_comment(state: ReviewState) -> dict:
    """TODO: post inline comments on the PR via the GitHub API."""
    for c in state.get("final_comments", []):
        print(f"[{c['severity']}] {c['path']}:{c['line']}  {c['issue']}")
        print(f"    suggestion: {c['suggestion']}")
    return {}


# Routing

def after_verify(state: ReviewState) -> str:
    """Send work back to the reviewer once if the verifier rejected everything."""
    if state.get("needs_retry") and state.get("attempts", 0) < MAX_ATTEMPTS:
        return "reviewer"
    return "post_comment"



In [8]:

# Wiring

def build_graph():
    g = StateGraph(ReviewState)
    g.add_node("fetch_diff", fetch_diff)
    g.add_node("retrieve", retrieve)
    g.add_node("reviewer", reviewer)
    g.add_node("verifier", verifier)
    g.add_node("post_comment", post_comment)

    g.add_edge(START, "fetch_diff")
    g.add_edge("fetch_diff", "retrieve")
    g.add_edge("retrieve", "reviewer")
    g.add_edge("reviewer", "verifier")
    g.add_conditional_edges("verifier", after_verify, ["reviewer", "post_comment"])
    g.add_edge("post_comment", END)
    return g.compile()


if __name__ == "__main__":
    app = build_graph()
    app.invoke({"repo": "project-practice/project/pr-review-test", "pr_number": 1})

[bug] utils.py:4  apply_discount now expects a 0 to 1 rate, but service.py still passes a percentage like 10, so orders will raise ValueError.
    suggestion: Update calculate_total to pass discount_percent / 100.
